# ResNet18: Train-Test Split Evaluation

## Objective

The objective of this notebook is to evaluate a transfer learning approach using the ResNet18 architecture.

Unlike the previous notebook, which focused on handcrafted features and classical machine learning models, this notebook fine-tunes a pretrained convolutional neural network on the custom image dataset.

The dataset is divided into:

- 80% Training Set
- 20% Test Set

The best performing model is saved and evaluated using standard classification metrics.

## Import Required Libraries

Import the libraries required for transfer learning, dataset loading, model training, evaluation, and visualization.

In [ ]:
import os
import copy
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

## Define Image Preprocessing Pipeline

Create separate preprocessing pipelines for training and testing.

The training pipeline includes data augmentation to improve generalization, while the testing pipeline only performs deterministic preprocessing.

In [ ]:
train_transform = transforms.Compose([

    transforms.Resize((256,256)),

    transforms.RandomCrop(224),

    transforms.RandomHorizontalFlip(0.5),

    transforms.ColorJitter(
        brightness=0.1,
        contrast=0.1
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )

])

test_transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )

])

## Load Dataset

Load the image dataset using the ImageFolder API and obtain the corresponding class labels for subsequent dataset splitting.

In [ ]:
dataset = datasets.ImageFolder("../dataset")

labels = dataset.targets

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(dataset.class_to_idx)

print(device)

## Create Train-Test Split

Split the dataset into training and testing subsets using an 80:20 ratio while preserving class balance through stratified sampling.

Separate data loaders are then created for efficient mini-batch training.

In [ ]:
indices = np.arange(len(dataset))

train_idx, test_idx = train_test_split(

    indices,

    test_size=0.20,

    random_state=42,

    stratify=labels

)

train_dataset = copy.deepcopy(dataset)
train_dataset.transform = train_transform

test_dataset = copy.deepcopy(dataset)
test_dataset.transform = test_transform

train_subset = Subset(train_dataset, train_idx)

test_subset = Subset(test_dataset, test_idx)

train_loader = DataLoader(
    train_subset,
    batch_size=8,
    shuffle=True
)

test_loader = DataLoader(
    test_subset,
    batch_size=8,
    shuffle=False
)

print(len(train_subset))
print(len(test_subset))

## Initialize the ResNet18 Model

Load the ImageNet pretrained ResNet18 model.

The final fully connected layer is replaced with a two-class classifier, enabling transfer learning on the screen recapture detection task.

In [ ]:
weights = models.ResNet18_Weights.DEFAULT

model = models.resnet18(weights=weights)

for param in model.parameters():
    param.requires_grad = False

for param in model.layer3.parameters():
    param.requires_grad = True

for param in model.layer4.parameters():
    param.requires_grad = True

model.fc = nn.Linear(512,2)

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

optimizer = optim.AdamW(

    filter(lambda p: p.requires_grad,
           model.parameters()),

    lr=5e-5,

    weight_decay=1e-4

)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(

    optimizer,

    mode="max",

    factor=0.5,

    patience=3

)

## Define Training and Evaluation Functions

Implement reusable functions for:

- Training one epoch
- Evaluating model performance
- Computing prediction accuracy

These functions simplify the training loop and improve code readability.

In [ ]:
def train_one_epoch():

    model.train()

    running_loss = 0

    correct = 0

    total = 0

    for images, labels in train_loader:

        images = images.to(device)

        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, preds = torch.max(outputs,1)

        total += labels.size(0)

        correct += (preds==labels).sum().item()

    return running_loss/len(train_loader), correct/total


def evaluate():

    model.eval()

    preds = []

    targets = []

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)

            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs,1)

            preds.extend(predicted.cpu().numpy())

            targets.extend(labels.cpu().numpy())

    acc = accuracy_score(targets,preds)

    return acc,preds,targets
    

## Train the Model

Train the ResNet18 model using the training dataset.

During training:

- Validation accuracy is monitored.
- Early stopping prevents overfitting.
- The best-performing checkpoint is stored for later evaluation.

In [ ]:
best_acc = 0

best_model = copy.deepcopy(model.state_dict())

counter = 0

PATIENCE = 5

EPOCHS = 40

for epoch in range(EPOCHS):

    loss,train_acc = train_one_epoch()

    val_acc,preds,targets = evaluate()

    scheduler.step(val_acc)

    print(
        f"Epoch {epoch+1:02d} | "
        f"Loss {loss:.4f} | "
        f"Train {train_acc:.4f} | "
        f"Test {val_acc:.4f}"
    )

    if val_acc > best_acc:

        best_acc = val_acc

        best_model = copy.deepcopy(model.state_dict())

        counter = 0

    else:

        counter += 1

    if counter >= PATIENCE:

        print("Early Stopping")

        break

In [ ]:
model.load_state_dict(best_model)

os.makedirs("../models",exist_ok=True)

torch.save(

    model.state_dict(),

    "../models/best_resnet18_80_20.pth"

)

print("Best Model Saved")

## Evaluate the Final Model

Evaluate the best model on the held-out test dataset.

Performance is reported using:

- Accuracy
- Precision
- Recall
- F1 Score
- Classification Report

These metrics provide an estimate of the model's ability to generalize to unseen images.

In [ ]:
acc,preds,targets = evaluate()

precision = precision_score(targets,preds)

recall = recall_score(targets,preds)

f1 = f1_score(targets,preds)

print(f"Accuracy : {acc:.4f}")

print(f"Precision : {precision:.4f}")

print(f"Recall : {recall:.4f}")

print(f"F1 Score : {f1:.4f}")

print()

print(classification_report(targets,preds))

# Summary

This notebook explored transfer learning using the ResNet18 architecture with an 80:20 train-test split.

### Training Strategy

- ImageNet pretrained ResNet18
- Transfer Learning
- Data Augmentation
- AdamW Optimizer
- Learning Rate Scheduling
- Early Stopping

### Evaluation

The trained model was evaluated on a completely unseen test set using standard classification metrics.

This experiment provides a realistic estimate of the model's generalization performance and serves as an additional benchmark alongside the Stratified 5-Fold Cross Validation experiments presented in the following notebooks.

The subsequent notebook investigates whether evaluating the model using Stratified 5-Fold Cross Validation can provide a more reliable estimate of its performance on the limited dataset.